In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

#DATA CLEANING & PREPROCESSING
def prepare_system(file_path):
    try:
        df = pd.read_csv(file_path)
    except:
        return print(f"Error: Please ensure '{file_path}' is uploaded to Colab.")

    #Fix shifted columns
    def fix_row(row):
        sports_keywords = ['Cricket', 'Football', 'Padel', 'Futsal', 'Swimming', 'Tennis', 'Gym', 'Basketball']
        io_val = str(row['Indoor / Outdoor'])
        if any(s in io_val for s in sports_keywords) and not ('Indoor' in io_val or 'Outdoor' in io_val):
            return pd.Series([f"{row['Area / Location']}, {row['Sports Offered']}", row['Indoor / Outdoor'], 'Indoor/Outdoor'],
                             index=['Area / Location', 'Sports Offered', 'Indoor / Outdoor'])
        return pd.Series([row['Area / Location'], row['Sports Offered'], row['Indoor / Outdoor']],
                         index=['Area / Location', 'Sports Offered', 'Indoor / Outdoor'])

    df[['Area / Location', 'Sports Offered', 'Indoor / Outdoor']] = df.apply(fix_row, axis=1)

    #Clean Pricing and Features
    def clean_p(val):
        nums = re.findall(r'\d+', str(val).replace(',', ''))
        return sum(map(float, nums)) / len(nums) if nums else 0.0

    df['Price_Num'] = df['Typical Hourly Rate (PKR)'].apply(clean_p)
    df['Pool_Bit'] = df['Swimming Pool'].apply(lambda x: 1 if str(x).lower() in ['yes', 'indoor', 'swimming pool'] else 0)
    df['Gym_Bit'] = df['Gym'].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)
    df['Area_Name'] = df['Area / Location'].apply(lambda x: str(x).split(',')[0].strip())

    #Preprocessing (Scaling & Encoding)
    df['Sports_List'] = df['Sports Offered'].apply(lambda x: [s.strip().title() for s in str(x).split(',')])
    mlb = MultiLabelBinarizer()
    sports_feat = pd.DataFrame(mlb.fit_transform(df['Sports_List']), columns=mlb.classes_, index=df.index)

    area_feat = pd.get_dummies(df['Area_Name'], prefix='Area')

    scaler = MinMaxScaler()
    num_feat = pd.DataFrame(scaler.fit_transform(df[['Price_Num', 'Pool_Bit', 'Gym_Bit']]),
                            columns=['Norm_Price', 'Norm_Pool', 'Norm_Gym'], index=df.index)

    features = pd.concat([num_feat, sports_feat, area_feat], axis=1)
    return df, features, mlb, scaler

#THE SMART RECOMMENDATION ENGINE
class SmartRecommender:
    def __init__(self, df, features, mlb, scaler):
        self.df, self.features, self.mlb, self.scaler = df, features, mlb, scaler

    def get_smart_recommendation(self, loc="", sport="", price="", gym="", pool="", n=5):
        #Build the User Vector
        user_vec = pd.Series(0.0, index=self.features.columns)
        input_count = 0

        #Map Location (Fuzzy match)
        if loc:
            match = [c for c in self.features.columns if loc.lower() in c.lower() and "Area_" in c]
            if match:
                user_vec[match[0]] = 1.0
                input_count += 1

        #Map Sport (Exact match)
        if sport:
            match = [c for c in self.mlb.classes_ if sport.title() == c]
            if match:
                user_vec[match[0]] = 1.0
                input_count += 1

        #Map Gym/Pool
        if gym.lower() == 'yes':
            user_vec['Norm_Gym'] = 1.0
            input_count += 1
        if pool.lower() == 'yes':
            user_vec['Norm_Pool'] = 1.0
            input_count += 1

        #Map Price
        if price and str(price).strip() not in ['', '-']:
            try:
                scaled_p = self.scaler.transform([[float(price), 0, 0]])[0,0]
                user_vec['Norm_Price'] = scaled_p
                input_count += 1
            except: pass

        #DECIDE MODEL BASED ON ENTRIES
        if input_count == 0:
            print("No preferences provided. Showing top general results...")
            return self.df.head(n), "None (Default View)"

        elif input_count == 1:
            print(f"Detected 1 requirement. Triggering KNN Model for precision...")
            knn = NearestNeighbors(n_neighbors=n, metric='euclidean')
            knn.fit(self.features)
            dists, indices = knn.kneighbors(user_vec.values.reshape(1, -1))
            res = self.df.iloc[indices[0]].copy()
            res['Match_Score_%'] = (1 / (1 + dists[0]) * 100).round(2)
            return res, "K-Nearest Neighbors (KNN)"

        else:
            print(f"Detected {input_count} requirements. Triggering Cosine Similarity Model for profile matching...")
            scores = cosine_similarity(user_vec.values.reshape(1, -1), self.features).flatten()
            top_idx = scores.argsort()[-n:][::-1]
            res = self.df.iloc[top_idx].copy()
            res['Match_Score_%'] = (scores[top_idx] * 100).round(2)
            return res, "Cosine Similarity"

#RUN INTERFACE
df_data, feat_matrix, mlb_tool, scale_tool = prepare_system('Data.csv')
engine = SmartRecommender(df_data, feat_matrix, mlb_tool, scale_tool)

print("KARACHI SMART COURT FINDER")
l = input("Location (e.g., DHA) [Leave blank for Any]: ").strip()
s = input("Sport (e.g., Cricket) [Leave blank for Any]: ").strip()
p = input("Budget (e.g., 4000) [Leave blank for Any]: ").strip()
g = input("Need Gym? (yes/no) [Leave blank for No]: ").strip()
pl = input("Need Pool? (yes/no) [Leave blank for No]: ").strip()

results, model_used = engine.get_smart_recommendation(l, s, p, g, pl)

print("\n" + "="*60)
print(f"RECOMMENDATIONS (Generated via {model_used})")
print("="*60)
print(results[['Facility Name', 'Area / Location', 'Sports Offered', 'Typical Hourly Rate (PKR)', 'Match_Score_%']])